# RAG Support Assistant remote benchmark

Run this notebook inside Google Colab. It keeps the local Windows machine and the iMac as thin clients only: no local Ollama, no local GraceKelly, no Docker, and no local model pulls.

The safe default cells run a mock provider benchmark first. The live direct-Mistral benchmark is opt-in inside the notebook and reads `MISTRAL_API_KEY` with `getpass`, so the key is not printed.

In [ ]:
# Runtime probe: this runs in Colab, not on the local/iMac machines.
import os
import platform
import subprocess

print(platform.platform())
print(platform.python_version())
print("CPU count:", os.cpu_count())
print("Memory:")
print("\n".join(open('/proc/meminfo', encoding='utf-8').read().splitlines()[:3]))
print("GPU:")
gpu = subprocess.run(['bash', '-lc', 'command -v nvidia-smi >/dev/null && nvidia-smi -L || true'], text=True, capture_output=True)
print(gpu.stdout.strip() or 'No GPU visible')

## Clone the repository

If the local branch contains commits that are not pushed, push them first or upload an archive manually. Colab can only clone what GitHub can see. The default branch below is `master`.

In [ ]:
REPO_URL = 'https://github.com/brownjuly2003-code/RAG_Support_Assistant.git'
BRANCH = 'master'

!rm -rf /content/RAG_Support_Assistant
!git clone --depth 1 --branch "$BRANCH" "$REPO_URL" /content/RAG_Support_Assistant
%cd /content/RAG_Support_Assistant
!git rev-parse --short HEAD
!git status --short --branch

In [ ]:
# Install project dependencies inside Colab. This may take several minutes.
# It does not install or start Ollama/GraceKelly/Docker.
%pip install -q -r requirements.txt

## Optional aircargo corpus setup

`data/uploads/aircargo/` is intentionally gitignored. For full-corpus aircargo/RAGAS work, provide an archive through `AIRCARGO_ARCHIVE_URL` or upload `aircargo_uploads.zip` / `aircargo_uploads.tar.gz` when prompted. The archive should contain either an `aircargo/` folder or the markdown files directly.

In [ ]:
from pathlib import Path
import os
import shutil
import tarfile
import urllib.request
import zipfile

project_root = Path('/content/RAG_Support_Assistant')
aircargo_dir = project_root / 'data' / 'uploads' / 'aircargo'
archive_url = os.environ.get('AIRCARGO_ARCHIVE_URL', '').strip()

def _assert_inside_extract_dir(path: Path, extract_dir: Path) -> None:
    root = extract_dir.resolve()
    target = path.resolve()
    if target != root and root not in target.parents:
        raise RuntimeError(f'Archive member escapes extract dir: {path}')

def _extract_zip_safely(archive_path: Path, extract_dir: Path) -> None:
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            _assert_inside_extract_dir(extract_dir / member.filename, extract_dir)
        archive.extractall(extract_dir)

def _extract_tar_safely(archive_path: Path, extract_dir: Path) -> None:
    with tarfile.open(archive_path) as archive:
        members = archive.getmembers()
        for member in members:
            _assert_inside_extract_dir(extract_dir / member.name, extract_dir)
        archive.extractall(extract_dir, members=members)

def _extract_aircargo_archive(archive_path: Path) -> None:
    extract_dir = project_root / '.tmp' / 'aircargo_uploads_extract'
    shutil.rmtree(extract_dir, ignore_errors=True)
    extract_dir.mkdir(parents=True, exist_ok=True)
    if archive_path.suffix == '.zip':
        _extract_zip_safely(archive_path, extract_dir)
    else:
        _extract_tar_safely(archive_path, extract_dir)
    candidates = [extract_dir / 'aircargo', extract_dir / 'uploads' / 'aircargo', extract_dir]
    source_dir = next((item for item in candidates if item.exists() and any(item.rglob('*.md'))), None)
    if source_dir is None:
        raise RuntimeError('Archive does not contain aircargo markdown files')
    shutil.rmtree(aircargo_dir, ignore_errors=True)
    aircargo_dir.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(source_dir, aircargo_dir)

if any(aircargo_dir.rglob('*.md')):
    print(f'Aircargo corpus already present: {len(list(aircargo_dir.rglob("*.md")))} markdown files')
elif archive_url:
    archive_path = project_root / '.tmp' / Path(archive_url).name
    archive_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(archive_url, archive_path)
    _extract_aircargo_archive(archive_path)
    print(f'Aircargo corpus restored: {len(list(aircargo_dir.rglob("*.md")))} markdown files')
else:
    from google.colab import files
    print('Upload aircargo_uploads.zip or aircargo_uploads.tar.gz to enable full aircargo runs.')
    uploaded = files.upload()
    if uploaded:
        archive_path = Path(next(iter(uploaded)))
        _extract_aircargo_archive(archive_path)
        print(f'Aircargo corpus restored: {len(list(aircargo_dir.rglob("*.md")))} markdown files')
    else:
        print('No upload received; aircargo reindex/live RAGAS cells should be skipped.')

## Optional aircargo reindex and RAGAS-style baseline

These cells run in Colab only. The mock RAGAS smoke does not call providers. The live cell is opt-in, uses direct Mistral through `getpass`, and starts with `--max-cases 3`.

In [ ]:
# Heavy Colab-only step: build the aircargo vector collection if the corpus was uploaded.
from pathlib import Path
aircargo_dir = Path('/content/RAG_Support_Assistant/data/uploads/aircargo')

if any(aircargo_dir.rglob('*.md')):
    %env VECTORDB_COLLECTION_PREFIX=rag_colab_aircargo
    %env RAG_SEMANTIC_CHUNKING=false
    %env RAG_STRUCTURAL_CHUNKING=false
    %env RAG_RERANKER_MODEL=cross-encoder/ms-marco-MiniLM-L-6-v2
    !python scripts/reindex.py --tenant aircargo
else:
    print('Aircargo corpus is missing; upload it before reindex.')

In [ ]:
# No-provider smoke for the aircargo RAGAS-style report writer.
!python scripts/aircargo_ragas_eval.py \
  --dataset evaluation/curated_cases_aircargo.jsonl \
  --tenant aircargo \
  --max-cases 3 \
  --seed 42 \
  --mock-runtime

In [ ]:
from getpass import getpass
import os

RUN_AIRCARGO_LIVE_RAGAS = False  # Set True only for the paid Colab live baseline.

if RUN_AIRCARGO_LIVE_RAGAS:
    if not os.environ.get('MISTRAL_API_KEY'):
        os.environ['MISTRAL_API_KEY'] = getpass('MISTRAL_API_KEY: ')
    !python scripts/aircargo_ragas_eval.py \
      --dataset evaluation/curated_cases_aircargo.jsonl \
      --tenant aircargo \
      --provider-target ministral-3b-latest \
      --max-cases 3 \
      --seed 42 \
      --allow-paid-apis
else:
    print('RUN_AIRCARGO_LIVE_RAGAS is False; live RAGAS-style baseline was skipped.')

In [ ]:
# Safe smoke: mock provider benchmark, no external provider calls and no DB persistence.
!python scripts/regression_eval.py \
  --baseline ministral-3b-latest \
  --candidate mistral-small-latest \
  --dataset evaluation/curated_cases.jsonl \
  --tenant all \
  --max-cases 3 \
  --seed 42 \
  --no-persist

## Optional live direct-Mistral benchmark

This cell spends API quota but does not use local model RAM. Keep `--max-cases` small for the first run. Do not use `ollama-small`, `gk-fast`, `gk-strong`, or `gracekelly-mixed` here; those imply local services or browser-backed orchestration.

In [ ]:
from getpass import getpass
import os

RUN_LIVE = False  # Change to True only when you intentionally want paid API calls from Colab.

if RUN_LIVE:
    if not os.environ.get('MISTRAL_API_KEY'):
        os.environ['MISTRAL_API_KEY'] = getpass('MISTRAL_API_KEY: ')
    !python scripts/regression_eval.py \
      --baseline ministral-3b-latest \
      --candidate mistral-small-latest \
      --dataset evaluation/curated_cases.jsonl \
      --tenant all \
      --max-cases 3 \
      --seed 42 \
      --allow-paid-apis \
      --no-persist
else:
    print('RUN_LIVE is False; live paid API benchmark was skipped.')

In [ ]:
# List generated reports for download from the Colab file browser.
!find reports/regression -maxdepth 1 -type f -printf '%TY-%Tm-%Td %TH:%TM %p\n' 2>/dev/null | sort | tail -20 || true